In [10]:
#r "..\src\functions\bin\Debug\net9.0\functions.dll"
#r "nuget: Azure.AI.OpenAI, 2.1.0"
#r "nuget: Azure.Core, 1.45.0"
#r "nuget: Azure.AI.Projects, 1.0.0-beta.2"

using Azure;
using Azure.AI.Projects;
using System.Text.Json;
using Azure.AI.OpenAI;
using Azure.Communication.Messages;
using OpenAI.Chat;

var endpoint = new Uri("https://marco-ma439rbt-eastus2.openai.azure.com/");
var model = "gpt-4.1-nano";
var deploymentName = "gpt-4.1-nano";
var apiKey = Environment.GetEnvironmentVariable("AZURE_OPENAI_API_KEY");

Console.WriteLine($"API Key lenght: {apiKey?.Length}");

AzureOpenAIClient azureClient = new(
    endpoint,
    new AzureKeyCredential(apiKey));
ChatClient chatClient = azureClient.GetChatClient(deploymentName);


Installed Packages Azure.AI.OpenAI, 2.1.0 Azure.AI.Projects, 1.0.0-beta.2 Azure.Core, 1.45.0

API Key lenght: 84


### Start a simple conversation

In [ ]:
List<ChatMessage> messages = new List<ChatMessage>()
{
    new SystemChatMessage("You are a helpful assistant."),
    new UserChatMessage("I am going to Paris, what should I see?"),
};

var options = new ChatCompletionOptions()
{
    MaxOutputTokenCount = 64,
    Temperature = 1.0f,
    TopP = 1.0f,
    FrequencyPenalty = 0.0f,
    PresencePenalty = 0.0f
};

var response = chatClient.CompleteChat(messages, options);
Console.WriteLine(response.Value.Content[0].Text);
"FINISHED"

### Use Tools/Actions

In [17]:
async Task<string> HandleToolCallAsync(ChatToolCall toolCall)
{
    if (toolCall.FunctionName == "GetWeather")
    {
        try
        {
            using JsonDocument argumentsDocument = JsonDocument.Parse(toolCall.FunctionArguments);

            if (argumentsDocument.RootElement.TryGetProperty("city", out JsonElement cityElement)
                && !string.IsNullOrEmpty(cityElement.GetString()))
            {
                return "The weather in " + cityElement.GetString() + " is sunny and 75 degrees.";
            }
            return "Invalid or missing 'city' argument.";
        }
        catch (JsonException ex)
        {
            // Handle JSON parsing errors
            return $"Error parsing JSON: {ex.Message}";
        }
    }

    return "Unknown function call.";
}

var json = "";
ChatTool getCurrentWeatherTool = ChatTool.CreateFunctionTool("GetWeather", "Get weather information for the provided city",
    BinaryData.FromString( json = 
        """
            {
              "type": "object",
              "properties": {
                "city": {
                  "type": "string",
                  "description": "The name of the city, e.g. Boston"
                }
              },
              "required": ["city"]
            }
        """
        )
);

Console.WriteLine(json);
// ChatTool scanBillTool = ChatTool.CreateFunctionTool("ScanBill", "Scans a bill and returns a summary and the total amount",
//     BinaryData.FromString(
//         """
//             {
//               "type": "object",
//               "properties": {
//                 "city": {
//                   "type": "string",
//                   "description": "The name of the city, e.g. Boston"
//                 }
//               },
//               "required": ["city"]
//             }
//         """
//         )
// );


var options = new ChatCompletionOptions() { MaxOutputTokenCount = 64, Tools = { getCurrentWeatherTool } };

List<ChatMessage> messages = new List<ChatMessage>()
{
    new SystemChatMessage("You are a helpful assistant."),
    new UserChatMessage("How is the weather in Paris?"),
};

var completion = await chatClient.CompleteChatAsync(messages, options);


// Handle different completion reasons
if (completion.Value?.FinishReason == ChatFinishReason.Stop)
{
    Console.WriteLine(completion.Value.Content[0].Text);
}
else if (completion.Value?.FinishReason == ChatFinishReason.ToolCalls)
{
    Console.WriteLine(completion.Value);

    // Handle tool call (e.g., OpenWeatherMap function)
    ChatToolCall toolCall = completion.Value.ToolCalls[0];
    string chatToolOutput = await HandleToolCallAsync(toolCall);
    //Console.WriteLine(chatToolOutput);
    messages.Add(new SystemChatMessage("The tool call was successful and returned 25C degrees. Describe the weather."));
    var completion2 = await chatClient.CompleteChatAsync(messages, options);
    Console.WriteLine(completion2.Value.Content[0].Text);

}


"FIN"
// var response = await chatClient.GetChatCompletionsAsync(
//     messages: yourMessagesList,
//     options: chatCompletionsOptions
// );

    {
      "type": "object",
      "properties": {
        "city": {
          "type": "string",
          "description": "The name of the city, e.g. Boston"
        }
      },
      "required": ["city"]
    }
OpenAI.Chat.ChatCompletion
The weather in Paris is currently 25°C. It seems to be warm and pleasant.


FIN

In [ ]:
var options = new ChatCompletionOptions() { MaxOutputTokenCount = 64 };

List<ChatMessage> messages = new List<ChatMessage>()
{
    new SystemChatMessage("""
You are Billy Assistant. Your role is to help the user pay a bill using a credit/debit card. The user might send the information in any order, so you need to be flexible and ask for the information you need to complete the payment.
You are running through whatsapp, the user might send the data in plain text or send pictures.
To help the user, you need to politely ask the user to provide the following information:

1. For a bill to scan. Be sure to obtain the TotalAmount, UserCode or OrderId. In case the bill is scanned, present the data obtained, and ask the user to confirm it.
2. A credit/debit card to use. Make sure to obtain complete and valid:
    CardNumber, ExpirationDate, Name, SecurityCode 
    In case the card is scanned, present the data obtained and ask the user to confirm it.

Do not ask the card if the user has not yet provided the bill to scan.
To validate the card expiration date, this is the current date: 2025-04-30 in format YYYY-MM-DD.
Use emojis to illustrate what you need.
If you summarize the information, present the information separated, bill, then card.
If the user asks for another information that is not related to the bill or the card, politely say that you do not know.
    """),
    // If the card has expired, ask for another card or correct the expiration date. 
    // new UserChatMessage("Hi what is your name?"),
    // new AssistantChatMessage ("I am Billy Assistant. First, could you please send me the bill to scan? Or Total Amount and your User Code or Order ID. 📄"),
    // new SystemChatMessage("User sent a picture of the bill, data extracted is TotalAmount: 25.00, UserCode: 32-66"),
    // new AssistantChatMessage ("Great! I see that the total amount is 25.00 and your user code is 32-66. Now, could you please provide me with your credit/debit card details? 💳"),
    // new SystemChatMessage("User sent a picture of the card, data extracted is CardNumber: 1234-5678-9012-3456, ExpirationDate: 12/25"),
    // new AssistantChatMessage ("Thank you! I see that the card number is 1234-5678-9012-3456 and the expiration date is 12/25. Could you please provide me with the security code (CVV) on the back of the card? 🔒"),
    // new UserChatMessage("Cant find the security code"),
    // new UserChatMessage("The security code is 123"),
};

var completion = await chatClient.CompleteChatAsync(messages, options);


// Handle different completion reasons
if (completion.Value?.FinishReason == ChatFinishReason.Stop)
{
    Console.WriteLine(completion.Value.Content[0].Text);
}else 
{
    Console.WriteLine("FinishReason " + completion.Value?.FinishReason);
}

Thank you for providing the security code. 😊

Could you please send me the bill details to scan? I need the total amount and either the UserCode or the OrderId to proceed. Once I have that, I can help you with the payment.
